# Sweep Analysis: f2fm6a3c

Analysis of the continual learning sweep comparing BP vs EFC across different task settings.

In [ ]:
import os
import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
SWEEP_ID = "ylutpmie"
WANDB_DIR = Path("./wandb")
SWEEP_DIR = WANDB_DIR / f"sweep-{SWEEP_ID}"

In [ ]:
def load_sweep_results(sweep_dir: Path, wandb_dir: Path) -> pd.DataFrame:
    """Load all run results from the local wandb logs for a given sweep."""
    results = []
    
    # Get all run IDs from sweep config files
    for config_file in sweep_dir.glob("config-*.yaml"):
        run_id = config_file.stem.replace("config-", "")
        
        # Find the corresponding run directory
        run_dirs = list(wandb_dir.glob(f"run-*-{run_id}"))
        if not run_dirs:
            print(f"Warning: No run directory found for {run_id}")
            continue
        
        run_dir = run_dirs[0]
        
        # Load config from sweep directory
        with open(config_file) as f:
            config = yaml.safe_load(f)
        
        # Load summary from run directory
        summary_file = run_dir / "files" / "wandb-summary.json"
        if not summary_file.exists():
            print(f"Warning: No summary file found for {run_id}")
            continue
            
        with open(summary_file) as f:
            summary = json.load(f)
        
        # Extract relevant info
        result = {
            "run_id": run_id,
            "setting": config.get("setting", {}).get("value"),
            "method": config.get("method", {}).get("value"),
            "cnn_pretrained": config.get("cnn_pretrained", {}).get("value"),
            "seed": config.get("seed", {}).get("value"),
            "final_avg_accuracy": summary.get("final_avg_accuracy"),
            "final_forgetting": summary.get("final_forgetting"),
        }
        results.append(result)
    
    return pd.DataFrame(results)

In [11]:
# Load all results
df = load_sweep_results(SWEEP_DIR, WANDB_DIR)
print(f"Loaded {len(df)} runs")
df.head()

Loaded 177 runs


,run_id,setting,method,cnn_pretrained,seed,final_avg_accuracy,final_forgetting
0,7ofc5jr5,TaskILTinyImageNet,efc,True,1,31.06,None
1,1sckvl0u,TaskILTinyImageNet,efc,True,1,37.42,None
2,a83vdysi,TaskILTinyImageNet,efc,True,1,36.28,None
3,xh0t9m9a,ClassILTinyImageNet10Task,efc,True,1,8.23,None
4,jn8krmkw,ClassILTinyImageNet10Task,efc,True,1,3.42,None


In [12]:
# Check data completeness
print("Runs per setting and method:")
print(df.groupby(["setting", "method"]).size().unstack(fill_value=0))

Runs per setting and method:
method                     efc
setting                       
ClassILTinyImageNet10Task   88
TaskILTinyImageNet          89


In [14]:
# Compute mean and std of final_avg_accuracy grouped by setting and method
accuracy_stats = df.groupby(["setting", "method"])["final_avg_accuracy"].agg(["mean", "std", "count"])
accuracy_stats = accuracy_stats.round(4)
accuracy_stats

,,mean,std,count
setting,method,,,
ClassILTinyImageNet10Task,efc,5.8901,2.1253,88
TaskILTinyImageNet,efc,28.4218,4.6055,89


In [15]:
# Create a clean results table: Setting vs Method with mean +/- std
# Note: final_avg_accuracy is already stored as percentage (0-100), not decimal (0-1)

# Pivot table with mean accuracy
pivot_mean = df.pivot_table(
    values="final_avg_accuracy", 
    index="setting", 
    columns="method", 
    aggfunc="mean"
)

# Pivot table with std
pivot_std = df.pivot_table(
    values="final_avg_accuracy", 
    index="setting", 
    columns="method", 
    aggfunc="std"
)

# Combine into formatted string (values are already percentages)
results_table = pivot_mean.copy()
for col in results_table.columns:
    results_table[col] = [
        f"{m:.2f}% +/- {s:.2f}%" 
        for m, s in zip(pivot_mean[col], pivot_std[col])
    ]

# Rename columns for display
results_table.columns = [col.upper() for col in results_table.columns]

# Order settings logically
setting_order = [
    "TaskILMNIST", "ClassILMNIST5Task",
    "TaskILCIFAR10", "ClassILCIFAR5Task",
    "TaskILTinyImageNet", "ClassILTinyImageNet10Task"
]
results_table = results_table.reindex(setting_order)

print("Final Average Accuracy (mean +/- std over 5 seeds)")
print("=" * 60)
results_table

Final Average Accuracy (mean +/- std over 5 seeds)


,EFC
setting,
TaskILMNIST,NaN
ClassILMNIST5Task,NaN
TaskILCIFAR10,NaN
ClassILCIFAR5Task,NaN
TaskILTinyImageNet,28.42% +/- 4.61%
ClassILTinyImageNet10Task,5.89% +/- 2.13%


In [ ]:
# Also create a numeric table for easier comparison
# Values are already percentages, no need to multiply by 100
numeric_table = pivot_mean.copy()
numeric_table.columns = [col.upper() for col in numeric_table.columns]
numeric_table = numeric_table.reindex(setting_order)
numeric_table = numeric_table.round(2)

print("Final Average Accuracy (%) - Numeric Values")
print("=" * 60)
numeric_table

Final Average Accuracy (%) - Numeric Values


,EFC
setting,
TaskILMNIST,NaN
ClassILMNIST5Task,NaN
TaskILCIFAR10,NaN
ClassILCIFAR5Task,NaN
TaskILTinyImageNet,28.42
ClassILTinyImageNet10Task,5.89


In [17]:
# Compute delta (EFC - BP) to see improvement
if "BP" in numeric_table.columns and "EFC" in numeric_table.columns:
    numeric_table["Delta (EFC - BP)"] = numeric_table["EFC"] - numeric_table["BP"]
    print("Accuracy Comparison with Delta")
    print("=" * 60)
    print(numeric_table)

In [18]:
# Forgetting analysis
forgetting_table = df.pivot_table(
    values="final_forgetting", 
    index="setting", 
    columns="method", 
    aggfunc="mean"
)
forgetting_table.columns = [col.upper() for col in forgetting_table.columns]
forgetting_table = forgetting_table.reindex(setting_order)
forgetting_table = forgetting_table.round(2)

print("Final Forgetting (mean over 5 seeds) - Lower is better")
print("=" * 60)
forgetting_table

Final Forgetting (mean over 5 seeds) - Lower is better


""
setting
TaskILMNIST
ClassILMNIST5Task
TaskILCIFAR10
ClassILCIFAR5Task
TaskILTinyImageNet
ClassILTinyImageNet10Task


In [19]:
# Compare CNN Pretrained (True) vs Non-Pretrained (False)
print("=" * 70)
print("CNN Pretrained vs Non-Pretrained Comparison")
print("=" * 70)

# Check how many runs have each pretrained value
print("\nRuns per cnn_pretrained value:")
print(df.groupby("cnn_pretrained").size())

# Pivot table: rows = (setting, method), columns = cnn_pretrained
pretrained_comparison = df.pivot_table(
    values="final_avg_accuracy",
    index=["setting", "method"],
    columns="cnn_pretrained",
    aggfunc=["mean", "std", "count"]
)

# Flatten column names
pretrained_comparison.columns = [f"{stat}_{pretrained}" for stat, pretrained in pretrained_comparison.columns]

# Calculate delta (pretrained=True minus pretrained=False)
if "mean_True" in pretrained_comparison.columns and "mean_False" in pretrained_comparison.columns:
    pretrained_comparison["delta_pretrained"] = (
        pretrained_comparison["mean_True"] - pretrained_comparison["mean_False"]
    )

pretrained_comparison = pretrained_comparison.round(2)
print("\nDetailed comparison by setting and method:")
print(pretrained_comparison)

# Create a cleaner summary table
print("\n" + "=" * 70)
print("Summary: Final Avg Accuracy by Pretraining Status")
print("=" * 70)

summary_pretrained = df.pivot_table(
    values="final_avg_accuracy",
    index="setting",
    columns=["method", "cnn_pretrained"],
    aggfunc="mean"
).round(2)

# Reorder settings
summary_pretrained = summary_pretrained.reindex(setting_order)
print(summary_pretrained)

# Overall effect of pretraining
print("\n" + "=" * 70)
print("Overall effect of pretraining (mean across all settings):")
print("=" * 70)
overall = df.groupby(["method", "cnn_pretrained"])["final_avg_accuracy"].agg(["mean", "std"]).round(2)
print(overall)

CNN Pretrained vs Non-Pretrained Comparison

Runs per cnn_pretrained value:
cnn_pretrained
True    177
dtype: int64

Detailed comparison by setting and method:
                                  mean_True  std_True  count_True
setting                   method                                 
ClassILTinyImageNet10Task efc          5.89      2.13          88
TaskILTinyImageNet        efc         28.42      4.61          89

Summary: Final Avg Accuracy by Pretraining Status
method                       efc
cnn_pretrained              True
setting                         
TaskILMNIST                  NaN
ClassILMNIST5Task            NaN
TaskILCIFAR10                NaN
ClassILCIFAR5Task            NaN
TaskILTinyImageNet         28.42
ClassILTinyImageNet10Task   5.89

Overall effect of pretraining (mean across all settings):
                        mean    std
method cnn_pretrained              
efc    True            17.22  11.85
